# 利用gensim使用word2vec简介
模型的论文：
  1. Tomas Mikolov.(2013). Distributed Representations of Words and Phrases and their Compositionality.
  2. Tomas Mikolov.(2013). Efficient Estimation of Word Representations in Vector Space.

其他参考：
 - https://blog.csdn.net/qq_30189255/article/details/103049569
 - https://blog.csdn.net/v_JULY_v/article/details/102708459
 - https://blog.csdn.net/yangbindxj/article/details/123911

早期开源的C语言版本，在Linux下可以编译测试： https://github.com/tmikolov/word2vec 869

In [1]:
from gensim.models import word2vec

语言模型的介绍
 - https://zhuanlan.zhihu.com/p/32292060
 - https://cloud.tencent.com/developer/article/2348463

## 1. 开箱使用

In [ ]:
# 示例文本数据
sentences = [
    ['我', '喜欢', '编程'],
    ['我', '喜欢', '旅游'],
    ['编程', '和', '旅游', '都', '是', '我的', '爱好']
]

In [ ]:
# 训练Word2Vec模型
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

word2vec模型，具体参数：
  1. sentences：可以是一个List，对于大语料集，建议使用BrownCorpus,Text8Corpus或·ineSentence构建。
  2. sg： 用于设置训练算法，默认为0，对应CBOW算法；sg=1则采用skip-gram算法。
  3. vector_size：是指输出的词的向量维数，默认为100。大的size需要更多的训练数据,但是效果会更好. 推荐值为几十到几百。
  4. window：为训练的窗口大小，8表示每个词考虑前8个词与后8个词（实际代码中还有一个随机选窗口的过程，窗口大小<=5)，默认值为5。
  5. alpha: 是学习速率
  6. seed：用于随机数发生器。与初始化词向量有关。
  7. min_count: 可以对字典做截断. 词频少于min_count次数的单词会被丢弃掉, 默认值为5。
  8. max_vocab_size: 设置词向量构建期间的RAM限制。如果所有独立单词个数超过这个，则就消除掉其中最不频繁的一个。每一千万个单词需要大约1GB的RAM。设置成None则没有限制。
  9. sample: 表示 采样的阈值，如果一个词在训练样本中出现的频率越大，那么就越会被采样。默认为1e-3，范围是(0,1e-5)
  10. workers:参数控制训练的并行数。
  11. hs: 是否使用HS方法，0表示不使用，1表示使用 。默认为0
  12. negative: 如果>0,则会采用negativesamp·ing，用于设置多少个noise words
  13. cbow_mean: 如果为0，则采用上下文词向量的和，如果为1（default）则采用均值。只有使用CBOW的时候才起作用。
  14. hashfxn： hash函数来初始化权重。默认使用python的hash函数
  15. iter： 迭代次数，默认为5。
  16. trim_rule： 用于设置词汇表的整理规则，指定那些单词要留下，哪些要被删除。可以设置为None（min_count会被使用）或者一个接受()并返回RU·E_DISCARD,uti·s.RU·E_KEEP或者uti·s.RU·E_DEFAU·T的函数。
  17. sorted_vocab： 如果为1（defau·t），则在分配word index 的时候会先对单词基于频率降序排序。
  18. batch_words：每一批的传递给线程的单词的数量，默认为10000569

In [1]:
# 获取词向量
word_vector = model.wv['编程']
print('词向量：', word_vector)

# 获取相似词
similar_words = model.wv.most_similar('编程', topn=3)
print('相似词：', similar_words)

词向量： [-8.6196875e-03  3.6657380e-03  5.1898835e-03  5.7419385e-03
  7.4669183e-03 -6.1676754e-03  1.1056137e-03  6.0472824e-03
 -2.8400505e-03 -6.1735227e-03 -4.1022300e-04 -8.3689485e-03
 -5.6000124e-03  7.1045388e-03  3.3525396e-03  7.2256695e-03
  6.8002474e-03  7.5307419e-03 -3.7891543e-03 -5.6180597e-04
  2.3483764e-03 -4.5190323e-03  8.3887316e-03 -9.8581640e-03
  6.7646410e-03  2.9144168e-03 -4.9328315e-03  4.3981876e-03
 -1.7395747e-03  6.7113843e-03  9.9648498e-03 -4.3624435e-03
 -5.9933780e-04 -5.6956373e-03  3.8508223e-03  2.7866268e-03
  6.8910765e-03  6.1010956e-03  9.5384968e-03  9.2734173e-03
  7.8980681e-03 -6.9895042e-03 -9.1558648e-03 -3.5575271e-04
 -3.0998408e-03  7.8943167e-03  5.9385742e-03 -1.5456629e-03
  1.5109634e-03  1.7900408e-03  7.8175711e-03 -9.5101865e-03
 -2.0553112e-04  3.4691966e-03 -9.3897223e-04  8.3817719e-03
  9.0107834e-03  6.5365066e-03 -7.1162102e-04  7.7104042e-03
 -8.5343346e-03  3.2071066e-03 -4.6379971e-03 -5.0889552e-03
  3.5896183e-03  5.

## 2. 利用更多的数据
采用网上的文本语料，语料大小将近100M，下载地址为http://mattmahoney.net/dc/text8.zip

In [25]:
# 数据可以逐行读写，这事一个测试
nline = 0
with open('text8', 'r', encoding='utf-8') as file:
    for line in file.readlines():
        nline += 1
    print(nline)

1


In [2]:
sentences = word2vec.Text8Corpus('text8')  # 将语料保存在sentence中

In [ ]:
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)  # 输出日志信息

In [3]:
# 利用预料sentences训练词向量空间模型
model = word2vec.Word2Vec(sentences, sg=1, vector_size=100,  window=5,  min_count=5,  negative=3, sample=0.001, hs=1, workers=4)  

In [4]:
model.save('text8_word2vec.model')   # 通过观察text8两个文件的时间戳，该训练花费约10分钟
#  model = word2vec.Word2Vec.load('text8_word2vec.model')

## 3. 测试词的查找

In [18]:
# 计算两个词的相似度/相关程度
word1 = 'man'
word2 = 'woman'
result1 = model.wv.similarity(word1, word2)
print(word1 + "和" + word2 + "的相似度为：", result1)

man和woman的相似度为： 0.6770231


In [10]:
# 计算某个词的相关词列表
word = 'bad'
result2 = model.wv.most_similar(word, topn=10)  # 10个最相关的
print("和" + word + "最相关的词有：")
for item in result2:
    print(item[0], item[1])

和bad最相关的词有：
good 0.7886669635772705
luck 0.7218429446220398
somebody 0.684140682220459
unpleasant 0.6598207354545593
feeling 0.6508694887161255
pretty 0.6479959487915039
feels 0.6445447206497192
strangest 0.6444723606109619
stupid 0.6435460448265076
funny 0.6402467489242554


In [23]:
# 寻找对应关系
# print(' "boy" is to "father" as "girl" is to ...? ')
model.wv.most_similar(['girl', 'father'], ['boy'], topn=3)

[('mother', 0.7679336667060852),
 ('wife', 0.7368452548980713),
 ('aunt', 0.6721643209457397)]

In [24]:
more_examples = ["she her he", "small smaller bad", "going went being"]
for example in more_examples:
    a, b, x = example.split()
    predicted = model.wv.most_similar([x, b], [a])[0][0]
    print("'%s' is to '%s' as '%s' is to '%s'" % (a, b, x, predicted))

'she' is to 'her' as 'he' is to 'his'
'small' is to 'smaller' as 'bad' is to 'worse'
'going' is to 'went' as 'being' is to 'was'


In [21]:
model.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=10)  # 10个最相关的

[('jagiellon', 0.6931903958320618),
 ('montferrat', 0.6912502646446228),
 ('sobieski', 0.6767150163650513),
 ('jogaila', 0.6566149592399597),
 ('queen', 0.6560076475143433),
 ('prince', 0.6558114290237427),
 ('valois', 0.6542955040931702),
 ('philippa', 0.6465786099433899),
 ('eldest', 0.6410372853279114),
 ('heir', 0.6349797248840332)]

In [12]:
# 寻找不合群的词
result5 = model.wv.doesnt_match("flower grass pig tree".split())
print("不合群的词：", result5)

不合群的词： tree


## 3. 增量训练

In [ ]:
more_sentences = [['Advanced', 'users', 'can', 'load', 'a', 'model', 'and', 'continue', 'training', 'it', 'with', 'more', 'sentences']]
model.build_vocab(more_sentences, update=True)

In [ ]:
model.train(more_sentences, total_examples=model.corpus_count, epochs=model.iter)

## 4. 中文语料

 - 实例参考： https://github.com/lzhenboy/word2vec-Chinese
 - 提供你自己的任意中文资料！